# 04 · Aşama 4 — Seçici tespit ve risk–kapsama analizi
Tanımlar (hata = saldırının NORMAL kararı alması veya normal örneğin ANOMALY kararı alması):
* **Kapsama (coverage)** = otomatik karar verilen pay (NORMAL veya ANOMALY)
* **Çekimser kalma oranı (abstention)** = UNCERTAIN payı (normal ve saldırı için ayrıca raporlanır)
* **Seçici risk (selective risk)** = otomatik kararlardaki hata oranı
* **Görülmemiş saldırı recall** = ANOMALY kararı alan görülmemiş saldırılar; **ertelenen** = UNCERTAIN alanlar
* Saldırı yaygınlığına göre yeniden ağırlıklandırma (π ∈ {0,001; 0,01; 0,1}) test kümesindeki temel oran etkisini kaldırır.

Önceden kaydedilmiş uç noktalar: **E2** — konformal bant, *eşleştirilmiş kapsama*da güçlü alternatiflerden
(B3 sezgisel bant, IF bandı, gizil skor bandı; rastgele çekimserlik yalnızca alt sınır kontrolüdür) daha iyi mi?
**E3** — görülmemiş saldırı ailelerinin aile bazında tespiti ve ertelenmesi.

In [ ]:
# ---- Kurulum (her defterde aynı): Drive'ı bağla, ortak kodu içe aktar ----
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
PROJECT_DIR = os.environ.get("TRUSTIDS_PROJECT", "/content/drive/MyDrive/TrustFed-TinyIDS-Final")
QUICK = os.environ.get("TRUSTIDS_QUICK", "0") == "1"   # hızlı mod YALNIZCA kodu test etmek içindir
# Devam modu: Colab oturumu kapanırsa, Drive'a kaydedilmiş tohumlar/adımlar atlanır ve kalan yerden devam edilir.
# Her şeyi baştan hesaplamak için: os.environ["TRUSTIDS_RESUME"] = "0"
RESUME = os.environ.get("TRUSTIDS_RESUME", "1") == "1"
sys.path.insert(0, PROJECT_DIR)

from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from trustids.config import Config
from trustids import utils

# Veri kümesi seçimi: varsayılan TON_IoT; ikinci veri kümesi için TRUSTIDS_CONFIG=edge_iiotset.json
CONFIG_NAME = os.environ.get("TRUSTIDS_CONFIG", "default.json")
CFG_PATH = Path(PROJECT_DIR) / "configs" / CONFIG_NAME
if CONFIG_NAME != "default.json" and not CFG_PATH.exists():
    raise FileNotFoundError(f"{CFG_PATH} bulunamadı: configs/ klasörünü de Drive'a yükleyin.")
cfg = Config.load(CFG_PATH) if CFG_PATH.exists() else Config(project_dir=PROJECT_DIR)
cfg.project_dir = PROJECT_DIR
if QUICK:
    cfg.train_seeds, cfg.validity_resplits, cfg.epochs = cfg.train_seeds[:2], 30, 15
    cfg.random_control_repeats, cfg.bootstrap_repeats = 10, 200
    cfg.lofo_seeds, cfg.fl_orig_rounds = cfg.lofo_seeds[:2], 3
    cfg.resolution_cal_windows, cfg.resolution_eval_windows = 600, 600
cfg.check()
ART = cfg.artifacts
FIG = Path(PROJECT_DIR) / cfg.figures_dir; FIG.mkdir(exist_ok=True)
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
print("Proje klasörü:", PROJECT_DIR, "| hızlı mod:", QUICK, "| devam modu:", RESUME)
print("Drive'da kayıtlı ilerleme:", utils.progress(cfg))

In [ ]:
from trustids import data as D, conformal as C, metrics as MT, evaluation as E, plotting as PL
P = D.load_prepared(ART / "prepared")
SEEDS = E.available_seeds(cfg); GROUPS = E.family_groups(cfg)
def decisions(seed):
    z = np.load(ART / "decisions" / f"seed{seed}.npz"); return {k: z[k] for k in z.files}
RUN = utils.new_run_dir(Path(PROJECT_DIR) / cfg.runs_dir, "midterm_phase4")
PREP_REP = utils.load_json(ART / "prepared" / "prepare_report.json")
if not PREP_REP["calibration_rule"]["rule_satisfied"]:          # önceden kaydedilmiş n_cal kuralı
    cfg.alpha = PREP_REP["calibration_rule"]["alpha_effective"]
EXCLUDE_MACRO = () if PREP_REP.get("mitm_in_macro_average", True) else ("mitm",)   # mitm yedek kuralı
utils.save_json(utils.environment_info(), RUN / "environment.json")
print("alpha =", cfg.alpha, "| makro ortalamadan çıkarılan:", EXCLUDE_MACRO)
print("sonuç klasörü:", RUN)

## Genel seçici metrikler (tohum 0), yaygınlığa göre yeniden ağırlıklandırma ile

In [ ]:
S0, D0 = E.load_scores(cfg, SEEDS[0]), decisions(SEEDS[0])
rows = []
for model in E.MODELS:
    ev = E.eval_set(P, S0, model)
    for pi in [None] + cfg.prevalence_grid:
        r = MT.selective_metrics(ev["y_attack"], D0[f"{model}__d"], pi)
        rows.append({"model": model, "yaygınlık": "test kümesi" if pi is None else pi, **r})
OVR = pd.DataFrame(rows); OVR.to_csv(RUN / "overall_selective_seed0.csv", index=False); OVR.round(4)

## Profesörün metrikleri: normal / bilinen / görülmemiş ayrı ayrı (tohum 0, AE)

In [ ]:
ev = E.eval_set(P, S0, "ae"); d = D0["ae__d"]; p = D0["ae__p"]
rows = []
for part, name in [("dev_test", "bilinen"), ("heldout_test", "görülmemiş")]:
    m = (ev["split"] == "benign_test") | (ev["split"] == part)
    y = ev["y_attack"][m]
    rows.append({"saldırılar": name, **MT.binary_metrics(y, ev["scores"][m], d[m] == C.ANOMALY),
                 **MT.selective_metrics(y, d[m])})
PROF = pd.DataFrame(rows); PROF.to_csv(RUN / "professor_metrics_seed0.csv", index=False); PROF.round(4)

## E2 — Konformal bant, eşleştirilmiş kapsamada alternatiflerden daha mı iyi? (tüm tohumlar)
Alternatif yöntemlere referans kuralla **aynı sayıda** UNCERTAIN kararı verdirilir (etiket kullanılmaz). `delta_sr > 0` konformal AE bandının seçici riskinin **daha düşük** olduğu anlamına gelir; fark yanlış alarm ve kaçırma bileşenlerine ayrıştırılır.

In [ ]:
rows = []
for seed in SEEDS:
    S, Dd = E.load_scores(cfg, seed), decisions(seed)
    ev = E.eval_set(P, S, "ae"); y = ev["y_attack"]; d_ref = Dd["ae__d"]
    h = C.heuristic_thresholds(ev["train"], cfg.heuristic_k[0], cfg.heuristic_k[1])
    alts = {"B3 sezgisel bant (eğitim hataları)": (ev["scores"], h["tau_H"])}
    for model, label in [("iforest", "B1 IF konformal bandı"), ("latent", "B5 gizil skor bandı")]:
        e2 = E.eval_set(P, S, model)
        alts[label] = (e2["scores"], C.thresholds(e2["cal"], cfg.alpha, cfg.beta)["tau_H"])
    for label, (s_alt, tauH) in alts.items():
        for pi in [None, 0.01]:
            r = MT.matched_coverage_comparison(y, d_ref, s_alt, tauH, pi)
            rows.append({"seed": seed, "comparison": label, "prevalence": "test kümesi" if pi is None else pi, **r})
    rc = MT.random_abstention_control(y, d_ref, cfg.random_control_repeats, seed)
    rows.append({"seed": seed, "comparison": "rastgele çekimserlik (alt sınır)", "prevalence": "test kümesi",
                 "sr_ref": rc["real_selective_risk"], "sr_alt": rc["random_mean"], "delta_sr": rc["delta"]})
E2 = pd.DataFrame(rows); E2.to_csv(RUN / "E2_matched_coverage.csv", index=False)
cols = ["abstain_ref", "abstain_alt", "sr_ref", "sr_alt", "delta_sr", "delta_false_alarm_part", "delta_miss_part"]
E2.groupby(["comparison", "prevalence"])[[c for c in cols if c in E2]].mean().round(5)

### E2 istatistikleri — iki yaygınlık düzeyinde
* **test kümesi** (önceden kaydedilmiş birincil analiz; eşleştirme etiketsiz, UNCERTAIN *sayısı* üzerinden);
* **π = 0,01** (keşifsel; eşleştirme yaygınlık ağırlıklı UNCERTAIN *kütlesi* üzerinden — etiket kullanır, yalnızca analiz amaçlıdır).

Not: 10 tohumla Wilcoxon testinin alabileceği en küçük iki yönlü p-değeri 2/1024 ≈ 0,002'dir; asıl kanıt
bootstrap güven aralıklarıdır. **Bu aralıklar ve testler yalnızca eğitim tohumu değişkenliğini ölçer** (tek, sabit bir
veri bölmesi üzerinde); test örneklerinin örnekleme belirsizliğini içermez.

In [ ]:
stats_rows = []
for prev, sub in E2.groupby("prevalence"):
    for comp, g in sub.groupby("comparison"):
        lo, hi = MT.bootstrap_ci(g["delta_sr"].to_numpy(), cfg.bootstrap_repeats, 0)
        stats_rows.append({"prevalence": prev, "comparison": comp, "mean_delta_sr": g["delta_sr"].mean(),
                           "ci_low": lo, "ci_high": hi, "seeds": len(g)})
ST = pd.DataFrame(stats_rows)
if len(SEEDS) >= 5:
    W = pd.concat([MT.paired_wilcoxon_holm(sub.assign(zero=0.0), "delta_sr", "zero", "comparison").assign(prevalence=prev)
                   for prev, sub in E2.groupby("prevalence")])
    ST = ST.merge(W[["prevalence", "comparison", "p_raw", "p_holm"]], on=["prevalence", "comparison"])
ST.to_csv(RUN / "E2_statistics.csv", index=False); ST.round(5)

### B3 sezgisel eşiğinin gerçekleşen yanlış alarm oranı (10 tohum)
μ + 3σ kuralı Gauss varsayımı altında yaklaşık %0,13 yanlış alarm *vaat eder*; ancak eğitim hatalarından
hesaplandığı için gerçek normal testte bu oran kontrol edilmez. Konformal kuralın oranı ise α ile sınırlıdır.

In [ ]:
rows = []
for seed in SEEDS:
    S, Dd = E.load_scores(cfg, seed), decisions(seed)
    ev = E.eval_set(P, S, "ae"); b = ~ev["y_attack"]
    h = C.heuristic_thresholds(ev["train"], cfg.heuristic_k[0], cfg.heuristic_k[1])
    rows.append({"tohum": seed, "B3_normal_ANOMALY": float(np.mean(ev["scores"][b] > h["tau_H"])),
                 "konformal_normal_ANOMALY": float(np.mean(Dd["ae__d"][b] == C.ANOMALY))})
B3F = pd.DataFrame(rows)
for col in ["B3_normal_ANOMALY", "konformal_normal_ANOMALY"]:
    lo, hi = MT.bootstrap_ci(B3F[col].to_numpy(), cfg.bootstrap_repeats, 0)
    print(f"{col}: ortalama {B3F[col].mean():.4f}  (%95 GA {lo:.4f}–{hi:.4f}) | Gauss 3σ nominal = 0.00135 | α = {cfg.alpha}")
B3F.to_csv(RUN / "B3_false_alarm_10seeds.csv", index=False)

## UNCERTAIN bandı gerçekte ne kazandırıyor? — katmanlama (10 tohum)
Üç yönlü kuralda ANOMALY ∪ UNCERTAIN birleşimi, **α+β düzeyindeki ikili kuralla aynıdır**; yani işaretlenen toplam
pay yapı gereği aynıdır. Kazanç, işaretlenen kümenin **anında alarm** ve **inceleme** katmanlarına bölünmesidir:
anında alarmların yanlış alarm yükü α ile, inceleme yükü β ile sınırlıdır. Tablo, katman başına kesinliği (precision)
gerçekçi yaygınlıkta (π = 0,01) ve 10 000 akış başına analist yükünü gösterir.

In [ ]:
tier_rows = []
for seed in SEEDS:
    S, Dd = E.load_scores(cfg, seed), decisions(seed)
    ev = E.eval_set(P, S, "ae")
    for pi in [None, 0.01]:
        for part, name in [("heldout_test", "görülmemiş"), ("dev_test", "bilinen")]:
            m = (ev["split"] == "benign_test") | (ev["split"] == part)
            t = MT.tiering_vs_binary(ev["y_attack"][m], Dd["ae__p"][m], cfg.alpha, cfg.beta, pi)
            t["tohum"], t["yaygınlık"], t["saldırılar"] = seed, "test kümesi" if pi is None else pi, name
            tier_rows.append(t)
TIER = pd.concat(tier_rows); TIER.to_csv(RUN / "tiering_vs_binary_10seeds.csv", index=False)
cols = ["benign_immediate_alarm", "benign_review", "attack_immediate_alarm", "attack_review", "attack_missed",
        "precision_alarm_tier", "precision_review_tier", "alarms_per_flows", "reviews_per_flows"]
TIER.groupby(["saldırılar", "yaygınlık", "rule"])[cols].mean().round(4)

## Risk–kapsama eğrileri (tohum 0)

In [ ]:
curves = {}
for model, label in [("ae", "AE konformal (β taranır)"), ("iforest", "IF konformal"), ("latent", "gizil skor konformal")]:
    ev_m = E.eval_set(P, S0, model)
    curves[label] = MT.risk_coverage_curve(ev_m["y_attack"], D0[f"{model}__p"], cfg.alpha)
ev = E.eval_set(P, S0, "ae"); h = C.heuristic_thresholds(ev["train"], *cfg.heuristic_k)
pts = []
for k in np.linspace(0, 0.4, 40):
    n_abs = int(k * len(ev["scores"]))
    tl = MT.band_for_abstention(ev["scores"], h["tau_H"], n_abs)
    pts.append(MT.selective_metrics(ev["y_attack"], C.decide_by_thresholds(ev["scores"], tl, h["tau_H"])))
curves["B3 sezgisel bant"] = pd.DataFrame(pts).sort_values("coverage")
ref = MT.selective_metrics(ev["y_attack"], D0["ae__d"])
PL.risk_coverage(curves, "Risk–kapsama (tohum 0)", RUN / "risk_coverage.png",
                 {f"önceden kaydedilmiş α={cfg.alpha}, β={cfg.beta}": (ref["coverage"], ref["selective_risk"])});

In [ ]:
# AURC: güven = p-değerinin karar sınırına (log ölçekte) uzaklığı.
# Not: AURC her modelin KENDİ kararları ve sıralaması üzerinden hesaplanır; modeller farklı karar kümeleri
# ürettiği için modeller arası karşılaştırmaya uygun değildir (yalnızca bilgi amaçlı, birincil uç nokta değildir).
rows = []
for model in E.MODELS:
    ev_m = E.eval_set(P, S0, model); p_m = D0[f"{model}__p"]
    pred = p_m <= cfg.alpha
    conf = np.abs(np.log(p_m) - np.log(cfg.alpha))
    rows.append({"model": model, **MT.aurc(ev_m["y_attack"], conf, pred == ev_m["y_attack"])})
AU = pd.DataFrame(rows); AU.to_csv(RUN / "aurc_seed0.csv", index=False); AU.round(5)

## Düşük p-değeri gerçekten 'saldırı' mı demek? (p-değeri aralığına göre güvenilirlik, tohum 0, AE)

In [ ]:
RB = MT.error_by_pvalue_bin(ev["y_attack"], D0["ae__p"], D0["ae__d"], cfg.pvalue_bins)
PL.reliability_bins(RB, "p-değeri aralığına göre saldırı payı ve karar hatası (AE, tohum 0)", RUN / "reliability_bins.png")
RB.round(4)

## E3 — Görülmemiş aileler: tespit edilen, ertelenen, kaçırılan (tohumlar üzerinden ortalama)
**Birincil recall** = çakışan (aynı vektörü normal etiketle de taşıyan) saldırılar *kaçırılmış* sayılarak hesaplanan kötümser değer.

In [ ]:
fam_rows = []
for seed in SEEDS:
    S, Dd = E.load_scores(cfg, seed), decisions(seed)
    ev = E.eval_set(P, S, "ae")
    t = MT.per_family_table(ev["types"], Dd["ae__d"], GROUPS, collision=ev["collision"],
                            exclude_from_macro=EXCLUDE_MACRO); t["seed"] = seed; fam_rows.append(t)
FAM = pd.concat(fam_rows)
E3_COLS = ["anomaly_rate_collisions_as_missed", "anomaly_rate", "uncertain_rate", "normal_rate"]   # ilki = birincil
E3 = FAM.groupby(["group", "family"])[[c for c in E3_COLS if c in FAM]].agg(["mean", "std"]).round(4)
FAM.to_csv(RUN / "E3_per_family_all_seeds.csv", index=False); E3

## İkincil: tekrar ağırlıklandırmasına duyarlılık (satır ağırlıklı ve benzersiz vektör, tohum 0, AE)
Garanti benzersiz vektörler içindir; satır (akış) ağırlıklı oranlar ikincil olarak, ek bir 'kayma' (S0) gibi raporlanır.

In [ ]:
ev = E.eval_set(P, S0, "ae"); d = D0["ae__d"]; w = ev["multiplicity"]
g = E.group_of(ev["types"], cfg)
rows = []
for grp in ["normal", "known", "held-out"]:
    m = g == grp
    rows.append({"grup": grp, "anomaly_benzersiz": np.mean(d[m] == C.ANOMALY),
                 "anomaly_satir_agirlikli": np.average(d[m] == C.ANOMALY, weights=w[m]),
                 "uncertain_benzersiz": np.mean(d[m] == C.UNCERTAIN),
                 "uncertain_satir_agirlikli": np.average(d[m] == C.UNCERTAIN, weights=w[m])})
MW = pd.DataFrame(rows); MW.to_csv(RUN / "multiplicity_weighted_seed0.csv", index=False); MW.round(4)

## Normal/saldırı çakışmaları (aynı vektör iki etiketle görülmüş)
Bir normal ve bir saldırı kopyası tutulur; saldırı kopyası işaretlenir. Recall: (a) olduğu gibi, (b) çakışmalar hariç, (c) çakışan saldırılar kaçırılmış sayılarak. **(c) birincil (kötümser) değerdir.**

In [ ]:
rows = []
for grp in ["known", "held-out"]:
    m = g == grp; c = ev["collision"] & m
    rec = np.mean(d[m] == C.ANOMALY)
    rec_excl = np.mean(d[m & ~ev["collision"]] == C.ANOMALY) if (m & ~ev["collision"]).any() else np.nan
    rec_pess = np.mean(np.where(ev["collision"][m], False, d[m] == C.ANOMALY))
    rows.append({"grup": grp, "çakışan_vektör": int(c.sum()), "recall": rec,
                 "recall_çakışmalar_hariç": rec_excl, "recall_çakışmalar_kaçmış (birincil)": rec_pess})
COL = pd.DataFrame(rows); COL.to_csv(RUN / "collisions_seed0.csv", index=False); COL.round(4)

## Ara rapor özeti (profesör için)

In [ ]:
val = pd.read_csv(FIG / "03_validity_E1.csv")
summary = {
    "yapilandirma": {"alpha": cfg.alpha, "beta": cfg.beta, "tohumlar": SEEDS, "gorulmeyecek_aileler": cfg.heldout_families},
    "E1_gecerlilik_tohum_ortalamasi": val.groupby(["model", "decision"])[["budget", "mean", "share_above_theory_q95"]].mean().round(5).reset_index().to_dict(orient="records"),
    "E2_eslestirilmis_kapsama": ST.round(5).to_dict(orient="records"),
    "B3_yanlis_alarm_10_tohum": B3F.drop(columns=["tohum"], errors="ignore").agg(["mean", "std"]).round(5).to_dict(),
    "katmanlama_10_tohum": TIER.groupby(["saldırılar", "yaygınlık", "rule"])[cols].mean().round(4).reset_index().to_dict(orient="records"),
    "E3_gorulmemis": FAM[FAM["group"] == "held-out"].groupby("family")[[c for c in E3_COLS if c in FAM]].mean().round(4).reset_index().to_dict(orient="records"),
    "profesor_metrikleri_tohum0": PROF.round(4).to_dict(orient="records"),
    "ortam": utils.environment_info(),
}
utils.save_json(summary, RUN / "midterm_summary.json")
import shutil
cfg.save(RUN / "config_used.json")
for f in ["data_file_info.json", "training_summary.json", "ae_selection.json", "environment.json"]:
    if (ART / f).exists():
        shutil.copy(ART / f, RUN / f)
shutil.copy(ART / "prepared" / "prepare_report.json", RUN / "prepare_report.json")
shutil.copytree(FIG, RUN / "figures", dirs_exist_ok=True)   # bu çalıştırmanın şekilleri/tabloları, ileride üzerine yazılmasın
print("Tüm sonuçlar kaydedildi (yapılandırma ve şekiller dahil):", RUN)
pd.DataFrame(summary["E3_gorulmemis"])

**Sonraki adım (final aşama):** `04b_uncertain_resolution`, ardından `F1_fl_teshis`, `F2_lofo`, `F3_asama5_9_on_tohum`.

In [ ]:
# Colab: Drive'a yazılan tüm dosyaların kaydedildiğinden emin ol (oturumu kapatmadan önce mutlaka çalıştırın)
print("Kaydedilen ilerleme:", utils.progress(cfg))
if IN_COLAB:
    drive.flush_and_unmount()
    print("Drive kaydedildi ve ayrıldı. Sonraki defter Drive'ı yeniden bağlar.")